# Guide 1: Neural networks: a function with knobs

*How LLMs work* · [Read the guide](https://www.vinaygautam.com/blog/how-llms-work/01-neural-networks) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/01-neural-networks.ipynb)

This notebook holds every code block from Guide 1, in the guide's order: 7 code cells, 6 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- A network is a function with knobs — no code
- Layers, and why the bend matters — no code
- The forward pass in shapes — no code
- Measuring wrongness: softmax and cross-entropy — no code
- Learning is walking downhill, precisely — no code
- Backpropagation: the chain rule, run backward — no code
- The whole loop in numpy — 7 code cells
- What carries forward to LLMs — no code

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## The whole loop in numpy

The whole loop is a few dozen lines of numpy, in blocks that run in order and share variables, like notebook cells. The first one turns the text into training examples: each row of `X` is a one-hot current character, and the matching entry of `y` is the index of the character that follows it.

In [2]:
import numpy as np

text = ("the cat sat on the mat and the dog sat on the log and the cat saw "
        "the dog then the dog ran to the cat and the cat ran on the mat")
chars = " abcdefghijklmnopqrstuvwxyz"
ids = np.array([chars.index(c) for c in text])  # (128,) one integer per character

X = np.eye(27)[ids[:-1]]  # (n, 27) one-hot current characters
y = ids[1:]               # (n,)    index of each next character
n = len(y)
print(X.shape, y.shape)

(127, 27) (127,)


The parameters start as small random numbers, with a fixed seed so your run matches this one. They can't all start at zero, because every hidden neuron would then compute the same thing and receive the same gradient, and the neurons could never become different from one another. The outputs shown here come from NumPy 2.2.6, the version pinned in the series `requirements.txt` as of October 2026. The biases are one-dimensional arrays, and numpy treats them as rows when it adds them.

In [3]:
rng = np.random.default_rng(0)
W1 = rng.normal(0, 0.01, (27, 16))  # (27, 16)
b1 = np.zeros(16)                   # (16,)
W2 = rng.normal(0, 0.01, (16, 27))  # (16, 27)
b2 = np.zeros(27)                   # (27,)
print(sum(a.size for a in (W1, b1, W2, b2)))

907


The forward pass and the loss are the formulas from earlier, applied to all $n$ rows at once.

In [4]:
def forward(X):
    h = np.maximum(0, X @ W1 + b1)        # (n, 16) hidden layer after ReLU
    z = h @ W2 + b2                       # (n, 27) logits
    z = z - z.max(axis=1, keepdims=True)  # same p, but exp can no longer overflow
    p = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)  # (n, 27) softmax
    return h, p

def loss_of(p):  # p must have one row per training example: it reads the correct answers y
    return -np.log(p[np.arange(n), y]).mean()  # mean cross-entropy

h, p = forward(X)
print(f"{loss_of(p):.3f}")

3.296


That's the $\ln 27$ of a uniform guess, as predicted. One line of `forward` subtracts each row's largest logit from the whole row. Softmax depends only on the differences between logits, so the probabilities don't change, and `exp` is never handed a large positive number. The backward pass is backpropagation written out one line per gradient.

In [5]:
def backward(X, h, p):
    dz = p.copy()               # (n, 27)
    dz[np.arange(n), y] -= 1    # predicted minus correct
    dz /= n                     # the loss is a mean over n examples
    dW2 = h.T @ dz              # (16, 27)
    db2 = dz.sum(axis=0)        # (27,)
    dh = dz @ W2.T              # (n, 16)
    dh[h <= 0] = 0              # ReLU: no gradient where the neuron was off
    dW1 = X.T @ dh              # (27, 16)
    db1 = dh.sum(axis=0)        # (16,)
    return dW1, db1, dW2, db2

Training repeats backward, update and forward 2,000 times with $\eta = 1$. A stride that long would bounce between $0$ and $6$ forever on the earlier one-knob loss. How long is too long depends on how sharply the particular loss curves, so a learning rate is found by trying.

In [6]:
eta = 1.0
print(0, f"{loss_of(p):.3f}")
for step in range(1, 2001):
    dW1, db1, dW2, db2 = backward(X, h, p)
    W1 -= eta * dW1
    b1 -= eta * db1
    W2 -= eta * dW2
    b2 -= eta * db2
    h, p = forward(X)
    if step in (100, 500, 1000, 2000):
        print(step, f"{loss_of(p):.3f}")

0 3.296
100 1.176
500 0.898
1000 0.890


2000 0.887


Average cross-entropy is lowest when those probabilities equal the true frequencies, which is a general property of this loss. Put 0.90 on "h" and 0.05 on each of the other two, for example, and weight each case by how often it happens: $\tfrac{11}{19}(-\ln 0.90) + \tfrac{7}{19}(-\ln 0.05) + \tfrac{1}{19}(-\ln 0.05) \approx 1.32$. The same sum with $\tfrac{11}{19}$, $\tfrac{7}{19}$ and $\tfrac{1}{19}$ as the probabilities comes to $0.84$. That is the floor for the 19 examples that follow "t", and every other character has a floor of its own. The next block builds the whole best-possible table from counts and measures its loss over all 127 examples.

In [7]:
counts = np.zeros((27, 27))                 # counts[a, b]: how often b follows a
np.add.at(counts, (ids[:-1], ids[1:]), 1)
best = X @ counts                           # (n, 27) each example's row of counts
best /= best.sum(axis=1, keepdims=True)     # (n, 27) frequencies
print(f"{loss_of(best):.3f}")

0.884


The trained network's probabilities sit close to the text's frequencies:

In [8]:
for c in "te":
    _, pc = forward(np.eye(27)[[chars.index(c)]])  # (1, 27)
    freq = counts[chars.index(c)] / counts[chars.index(c)].sum()  # (27,)
    top = np.argsort(pc[0])[::-1][:2]
    print(repr(c), "->", ", ".join(f"{chars[i]!r} {pc[0, i]:.2f} (text {freq[i]:.2f})" for i in top))

't' -> 'h' 0.62 (text 0.58), ' ' 0.32 (text 0.37)
'e' -> ' ' 0.90 (text 0.91), 'n' 0.10 (text 0.09)


---

[Guide 2: Tokenization](02-tokenizer.ipynb) →

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).